In [1]:
from open_dataset_store import quick_start
import pandas as pd
import numpy as np

import pandas as pd


In [2]:
store = quick_start('.', backend='local')
sum = store.summary()


Store initialised at: . (Backend: local)
📊 Dataset Store Summary
Base Directory : .
Backend        : local
------------------------------
Entities (Total: 2)
  - zones: 2
------------------------------
Entries (Total: 6)
  - experiments: 6


In [7]:
store.list_entries('experiments')

,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,processed_metadata
0,entry_0001,zone_001,1783344483,Senosr Data Test,raw_data/experiments/entry_0001_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
1,entry_0002,zone_001,1783344565,"Test CO2 variation, single person come in and ...",raw_data/experiments/entry_0002_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
2,entry_0003,zone_002,1785482758,Test 1,raw_data/experiments/entry_0005_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
3,entry_0004,zone_002,1785483118,Test 2,raw_data/experiments/entry_0006_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
4,entry_0005,zone_002,1785483802,Test 3,raw_data/experiments/entry_0005_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
5,entry_0006,zone_002,1785484346,Test 3,raw_data/experiments/entry_0006_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
6,entry_0007,zone_002,1785507025,Test 4,raw_data/experiments/entry_0007_zone_002_17855...,{},NaN


In [8]:
csv_path = "./Day 4 Test 5_2026-07-31.csv"

manual_occupancy = {
    "15:25:00": 2,
    "15:35:00": 1,
    "15:50:00": 0,
    "15:55:00": 2,
    "16:05:00": 1,
    "16:10:00": 0,
}

In [9]:
import pandas as pd


df_raw = pd.read_csv(csv_path)

def add_occupancy_and_localize_time(df, occupancy_schedule, tz='Asia/Kolkata'):
    """
    Converts 'timestamp' to local time and injects a forward-filled 
    'actual_occupancy' column based on a provided manual schedule.
    """
    # 1. Ensure timestamp is timezone aware and convert to local timezone
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['timestamp'] = df['timestamp'].dt.tz_convert(tz)
    
    # 2. Create a schedule dataframe
    schedule_df = pd.DataFrame(list(occupancy_schedule.items()), columns=['time_str', 'actual_occupancy'])
    
    # Extract the base date from the dataset (assuming 1-day experiment)
    exp_date = df['timestamp'].dt.date.iloc[0]
    
    # [FIXED] Use str(exp_date) instead of exp_date.astype(str)
    schedule_df['timestamp'] = pd.to_datetime(str(exp_date) + ' ' + schedule_df['time_str'])
    schedule_df['timestamp'] = schedule_df['timestamp'].dt.tz_localize(tz)
    schedule_df = schedule_df.sort_values('timestamp')
    
    # 3. Sort main data and merge using nearest backward match (holds previous value)
    df = df.sort_values('timestamp')
    df = pd.merge_asof(df, schedule_df, on='timestamp', direction='backward')
    
    # 4. Fill any timestamps that occurred before the first schedule entry with 0
    df['actual_occupancy'] = df['actual_occupancy'].fillna(0).astype(int)
    
    return df

df_processed = add_occupancy_and_localize_time(df_raw, manual_occupancy)
display(df_processed.head())


,timestamp,outside_t,outside_h,outside_c,outside_p,outside_a,outside_v,room_1_t,room_1_h,room_1_c,...,heated_a,heated_v,mixer,fan,flowrate,coolerState,heaterState,humidifierState,time_str,actual_occupancy
0,2026-07-31 15:19:56.699000+05:30,31.9,62.7,484,0,1,64,23.2,61.3,0,...,0,0,100,55,0,0,0,0,NaN,0
1,2026-07-31 15:20:01.702000+05:30,31.9,62.6,498,0,2,71,23.2,61.3,0,...,0,0,100,55,0,0,0,0,NaN,0
2,2026-07-31 15:20:06.700000+05:30,31.9,62.6,498,0,2,71,23.2,61.3,0,...,0,0,100,55,0,0,0,0,NaN,0
3,2026-07-31 15:20:11.700000+05:30,32.0,62.7,511,0,2,78,23.1,61.3,0,...,0,0,100,55,0,0,0,0,NaN,0
4,2026-07-31 15:20:16.699000+05:30,32.0,62.7,512,0,2,78,23.1,61.3,0,...,0,0,100,55,0,0,0,0,NaN,0


In [10]:
entry_id = store.create_entry_from_df(
    entry_type="experiments",
    df=df_processed,
    entity_id="zone_002",
    description="Test 4",
)



✅ Entry 'entry_0008' created. File saved as entry_0008_zone_002_1785508506_data.csv


In [14]:
store.delete_entry(entry_id='entry_0007', entry_type="experiments",)

  - Deleted raw file: raw_data/experiments/entry_0007_zone_002_1785506267_data.csv
  - Deleted processed [controller_evaluation]: processed_data/experiments/controller_evaluation/entry_0007_controller_evaluation.parquet
✅ Entry 'entry_0007' completely removed.


'entry_0007'